# Mosec

A high-performance, framework-agnostic model-serving library that wraps your Python
inference code behind a Rust web/runtime layer with automatic dynamic batching,
multi-stage pipelines, and multi-process parallelism.

- **Repo:** https://github.com/mosecorg/mosec (Apache-2.0)
- **Docs:** https://mosecorg.github.io/mosec/
- **One-liner:** "Bring your own model, get a production HTTP server with batching for free." 

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction <a id="introduction"></a>

Mosec is a model-serving framework built by the MOSEC team. The web server, request
queue, dynamic batcher, and inter-process communication are written in **Rust**, while
the model logic you write stays in **pure Python**. You subclass a `Worker`, implement
`forward()`, and Mosec turns it into an HTTP service that batches incoming requests and
runs your workers across multiple processes.

### What is it?

Mosec is a *thin, fast serving layer* for any Python ML model. It does **not** care
which framework you use (PyTorch, TensorFlow, JAX, ONNX Runtime, scikit-learn, a custom
NumPy function, or an LLM). Its job is the part that is annoying to build well by hand:

- accepting HTTP requests concurrently,
- **grouping them into dynamic batches** to maximize GPU/CPU throughput,
- routing batches through a **pipeline of stages** (e.g. CPU preprocess → GPU infer → CPU postprocess),
- running each stage in its **own process pool** so the Python GIL is not a bottleneck,
- exposing **Prometheus metrics** and structured logging.

### Why use it?

- **Free dynamic batching** — the single biggest throughput win for GPU inference, with one config flag (`max_batch_size`).
- **Framework-agnostic** — if it runs in Python, Mosec can serve it. No model-format conversion.
- **Low overhead** — the Rust hot path (HTTP, queueing, batching, IPC via shared memory) keeps tail latency low and avoids Python web-server bottlenecks.
- **True parallelism** — each stage runs in separate OS processes (`num=N`), sidestepping the GIL; mix CPU-heavy and GPU-heavy stages with independent replica counts.
- **Tiny, explicit API** — a `Worker` class and a `Server`. No DSL, no YAML model spec to learn.

### When to use it?

- You have a **custom Python inference pipeline** (pre/post-processing around a model) and want batching + scaling without writing a server.
- You serve **GPU models** and need dynamic batching but find Triton/TF-Serving too heavy or too rigid about model formats.
- You want **one process pool for CPU stages and another for the GPU stage**, tuned independently.
- You're standing up an **embedding / vision / ASR / LLM** endpoint and want msgpack or SSE streaming with minimal code.

## Key Features <a id="key-features"></a>

### Core capabilities

| Feature | Description | Why it matters |
|---------|-------------|----------------|
| **Dynamic batching** | Server collects requests up to `max_batch_size` or `max_wait_time` ms, then calls `forward` once with the whole batch | Turns many small GPU calls into a few large ones → much higher throughput |
| **Multi-stage pipeline** | Chain workers with `append_worker`; output of one stage feeds the next | Run CPU preprocessing and GPU inference in separate, independently-scaled pools |
| **Multi-process workers** | `num=N` spawns N OS processes per stage | Real parallelism past the GIL; saturate multi-core CPUs / multiple GPUs |
| **Rust runtime** | HTTP, request queue, batching scheduler, and IPC are native code | Low latency overhead and high concurrency without Python web-server limits |
| **Shared-memory IPC** | Large payloads (tensors/images) passed between Rust and workers via shared memory | Avoids serialization copies for big arrays |
| **Pluggable (de)serialization** | Default JSON; `MsgpackMixin` for msgpack; `TypedMsgPackMixin` for pydantic-validated input | Compact binary protocol for tensors; typed request validation |
| **SSE streaming** | Workers can emit incremental events with `send_stream_event` | Token-by-token LLM streaming over Server-Sent Events |
| **Built-in observability** | Prometheus `/metrics` endpoint, structured logging via `get_logger()` | Drop-in monitoring without extra middleware |

## Architecture Overview <a id="architecture"></a>

```
                 HTTP clients
                      │  POST /inference
                      ▼
        ┌─────────────────────────────┐
        │     Rust Controller         │   (one process)
        │  • HTTP server (routes)     │
        │  • request queue            │
        │  • DYNAMIC BATCHER          │   groups requests:
        │  • IPC over shared memory   │   batch when full OR max_wait_time hit
        └─────────────────────────────┘
            │            │            │
   stage 1  ▼   stage 2  ▼   stage 3  ▼     (each stage = its own process pool)
  ┌──────────────┐ ┌──────────────┐ ┌──────────────┐
  │ Preprocess   │ │ Inference    │ │ Postprocess  │
  │ Worker xN    │→│ Worker xM    │→│ Worker xK    │
  │ (CPU)        │ │ (GPU)        │ │ (CPU)        │
  └──────────────┘ └──────────────┘ └──────────────┘
```

### Components

1. **Rust controller** — a single supervisor process that runs the HTTP server,
   maintains the request queue, performs **dynamic batching**, and shuttles data
   to/from Python workers over shared-memory IPC. You never write Rust.
2. **Worker** — a Python class you subclass. Each instance runs in its own process.
   You implement `forward(self, data)`; optionally `deserialize`, `serialize`, and
   `__init__` (load weights here). Batched stages receive a **list** in `forward`.
3. **Server** — the Python entry point. You `append_worker(...)` each stage in order
   and call `server.run()`. The Server tells the Rust controller the pipeline topology
   (stages, replica counts, batch sizes).
4. **Stage pipeline** — workers are connected as a directed chain. Each stage has its
   own `num` (process count) and `max_batch_size`, so a slow CPU stage can be scaled
   wider than a single GPU stage.

## Installation <a id="installation"></a>

### Prerequisites

- **Python 3.8+** (CPython; Linux/macOS — Mosec ships prebuilt wheels with the Rust runtime).
- Your model's own deps (PyTorch / TF / ONNX Runtime / etc.).
- For GPU serving: the matching CUDA-enabled framework build and a GPU-visible host.
- Optional: `msgpack` and `pydantic` (pulled in by Mosec's mixins) for binary/typed I/O.

### Installation steps

Mosec is a single pip install — the Rust runtime comes inside the wheel, no toolchain needed.

In [ ]:
# Uncomment to install
# !pip install mosec
#
# Optional extras used in this notebook:
# !pip install mosec[mixin]   # msgpack + pydantic helpers
# !pip install numpy pillow   # for the worked examples below

# Verify the install (run where mosec is installed):
# import mosec
# print(mosec.__version__)

## Basic Usage <a id="basic-usage"></a>

### Quick start

A Mosec service is a Python script. Define a `Worker`, register it on a `Server`, and
run. Below is a minimal CPU service that squares numbers — the same shape you'd use for
a real model, just with `np`/`torch` work inside `forward`.

Save this as `server.py` and launch it with `python server.py`. It serves on
`http://0.0.0.0:8000` by default; send inference requests to `POST /inference`.

In [ ]:
# server.py  —  minimal Mosec service
from mosec import Server, Worker, get_logger

logger = get_logger()


class Square(Worker):
    """One worker process. `forward` runs per request (or per batch if batched)."""

    def forward(self, data: dict) -> dict:
        # Default (de)serialization is JSON, so `data` is already a Python dict.
        x = data["x"]
        return {"x": x, "square": x * x}


if __name__ == "__main__":
    server = Server()
    # num=2 → two worker processes for parallelism across requests.
    server.append_worker(Square, num=2)
    server.run()


Call it from a client once it's running:

```bash
# Start the server in one terminal:
python server.py --port 8000

# Query it from another:
curl -X POST http://127.0.0.1:8000/inference \
     -H 'Content-Type: application/json' \
     -d '{"x": 9}'
# -> {"x": 9, "square": 81}
```

Useful CLI flags (parsed by `Server`): `--address 0.0.0.0`, `--port 8000`,
`--timeout 10000` (per-request ms), `--debug` (verbose logs, no multiprocessing
fork tricks). Health/readiness lives at `GET /`, metrics at `GET /metrics`.

In [ ]:
# Programmatic client (equivalent to the curl above)
import json
import urllib.request

def call(payload: dict, url: str = "http://127.0.0.1:8000/inference") -> dict:
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())

# call({"x": 9})  ->  {"x": 9, "square": 81}   (run while server.py is up)


## Advanced Features <a id="advanced-features"></a>

### 1. Dynamic batching

Set `max_batch_size` (and optionally `max_wait_time` in ms). When batching is enabled,
`forward` receives a **list** of inputs and must return a list of the same length and
order. The Rust batcher forms a batch as soon as either the size limit is reached or the
wait window elapses.

In [ ]:
# Batched GPU-style inference. `forward` now takes & returns a LIST.
from mosec import Server, Worker
import numpy as np


class BatchedModel(Worker):
    def __init__(self):
        # Load weights once per process here (not per request).
        self.weight = np.random.rand(8, 8).astype("float32")

    def forward(self, data: list[dict]) -> list[dict]:
        # Stack the batch, run one matmul for the whole batch, then split back out.
        batch = np.array([d["vec"] for d in data], dtype="float32")  # (B, 8)
        out = batch @ self.weight                                    # (B, 8)
        return [{"out": row.tolist()} for row in out]


if __name__ == "__main__":
    server = Server()
    server.append_worker(
        BatchedModel,
        num=1,                 # 1 process (e.g. one GPU)
        max_batch_size=32,     # group up to 32 requests
        max_wait_time=10,      # ...but wait at most 10ms to fill the batch
    )
    server.run()


### 2. Multi-stage pipelines

Chain `append_worker` calls in pipeline order. Each stage runs in its own process pool
with independent `num` and `max_batch_size`. A common pattern: cheap CPU preprocessing
fanned out wide, one expensive GPU stage in the middle, cheap CPU postprocessing after.

In [ ]:
from mosec import Server, Worker
import numpy as np


class Preprocess(Worker):
    """CPU: parse/normalize input. Scaled wide because it's the bottleneck stage."""
    def forward(self, data: dict) -> dict:
        arr = np.asarray(data["pixels"], dtype="float32") / 255.0
        return {"tensor": arr.tolist()}


class Inference(Worker):
    """GPU: the model. Batched, single replica (one GPU)."""
    def __init__(self):
        self.model = lambda x: x.mean(axis=-1, keepdims=True)  # stand-in for a real net

    def forward(self, data: list[dict]) -> list[dict]:
        batch = np.array([d["tensor"] for d in data], dtype="float32")
        preds = self.model(batch)
        return [{"score": float(p)} for p in preds]


class Postprocess(Worker):
    """CPU: format the response."""
    def forward(self, data: dict) -> dict:
        return {"label": "positive" if data["score"] > 0.5 else "negative",
                "score": data["score"]}


if __name__ == "__main__":
    server = Server()
    server.append_worker(Preprocess, num=4)                       # 4 CPU procs
    server.append_worker(Inference, num=1, max_batch_size=32)     # 1 GPU proc, batched
    server.append_worker(Postprocess, num=2)                      # 2 CPU procs
    server.run()


### 3. Binary & typed I/O (msgpack + pydantic)

JSON is fine for small payloads but slow and lossy for tensors. Mix in `MsgpackMixin`
for a compact binary protocol, or `TypedMsgPackMixin` to get **pydantic validation**:
declare the request schema as a type hint on `forward` and Mosec rejects malformed input
with a `422` automatically.

In [ ]:
from mosec import Server, Worker
from mosec.mixin import TypedMsgPackMixin
from pydantic import BaseModel


class Request(BaseModel):
    text: str
    top_k: int = 5


class Embed(TypedMsgPackMixin, Worker):
    # Type hint drives validation + msgpack (de)serialization.
    def forward(self, req: Request) -> dict:
        # `req` is a validated pydantic instance; bad input never reaches here.
        vec = [float(ord(c) % 7) for c in req.text][: req.top_k]
        return {"embedding": vec}


if __name__ == "__main__":
    server = Server()
    server.append_worker(Embed, num=2)
    server.run()

# Use msgpack on the client side and set Content-Type: application/msgpack.
# For plain binary without pydantic, subclass `MsgpackMixin` instead.


### 4. SSE streaming (LLM token output)

For generative models, a worker can emit incremental events with `send_stream_event`
instead of returning one blob. The client receives Server-Sent Events, enabling
token-by-token streaming. The streaming worker must be the **last** stage in the
pipeline.

In [ ]:
from mosec import Server, Worker


class Generate(Worker):
    def forward(self, data: dict) -> str:
        prompt = data["prompt"]
        # Pretend-stream tokens; a real impl loops over model.generate(...).
        for i, tok in enumerate(prompt.split()):
            self.send_stream_event(tok + " ", index=0)  # index identifies the stream
        return ""  # final return closes the stream


if __name__ == "__main__":
    server = Server()
    server.append_worker(Generate, num=1)
    server.run()

# Client: POST to /inference with header `Accept: text/event-stream`
# and read the streamed `data:` chunks.


### 5. Custom (de)serialization & validation errors

Override `deserialize` / `serialize` to control the wire format yourself, and raise
`mosec.errors.ValidationError` to return a clean `422` for bad requests instead of a
`500`.

In [ ]:
from mosec import Server, Worker
from mosec.errors import ValidationError


class RawBytesWorker(Worker):
    def deserialize(self, data: bytes):
        # e.g. decode a raw image; raise ValidationError -> HTTP 422 for the caller.
        if not data:
            raise ValidationError("empty request body")
        return data

    def forward(self, data: bytes) -> dict:
        return {"num_bytes": len(data)}

    def serialize(self, data: dict) -> bytes:
        import json
        return json.dumps(data).encode()


if __name__ == "__main__":
    server = Server()
    server.append_worker(RawBytesWorker)
    server.run()


## Use Cases <a id="use-cases"></a>

#### Use Case 1: High-throughput embedding service

- **Context:** A semantic-search backend needs to embed thousands of short texts/sec on a single GPU.
- **Implementation:** One `Inference` worker with `max_batch_size=64`, `max_wait_time=5`, msgpack I/O via `MsgpackMixin`. A CPU `Tokenize` stage at `num=8` feeds it.
- **Results:** Dynamic batching turns scattered single-text calls into full GPU batches, multiplying throughput while the 5ms wait keeps p99 latency tight.

#### Use Case 2: Vision pipeline (decode → infer → NMS)

- **Context:** Object-detection API receives JPEG bytes and must return boxes.
- **Implementation:** `Decode` (CPU, `num=6`, Pillow) → `Detector` (GPU, batched) → `PostNMS` (CPU, `num=4`). Shared-memory IPC moves the decoded tensors without copy overhead.
- **Results:** CPU decode/NMS are scaled independently of the single GPU stage, so the GPU stays saturated rather than blocked on image decoding.

#### Use Case 3: Streaming LLM endpoint

- **Context:** A chat product wants token streaming without standing up Triton/TGI.
- **Implementation:** A single `Generate` worker that calls `send_stream_event` per token, served over SSE; `--timeout` raised to accommodate long generations.
- **Results:** Minimal code path to a streaming endpoint; pairs the model with Mosec's batching for concurrent sessions.

## Best Practices <a id="best-practices"></a>

1. **Load weights in `__init__`, not `forward`.** `__init__` runs once per worker process; `forward` runs on the hot path. Loading a model per request is the classic latency killer.
2. **Separate CPU and GPU work into stages.** Put tokenization/decoding/NMS in their own CPU stages with higher `num`, and keep one batched GPU stage. This keeps the GPU fed.
3. **Tune `max_batch_size` against `max_wait_time`.** Bigger batches raise throughput but add latency; the wait window bounds the worst-case delay when traffic is light. Start at `max_wait_time=10` and measure.
4. **Use msgpack/`TypedMsgPackMixin` for tensor or structured payloads.** JSON is the right default only for tiny inputs; binary protocols cut serialization cost and pydantic gives you free `422` validation.
5. **Set `num` per stage from a profile, not a guess.** Match CPU-stage `num` to available cores and GPU-stage `num` to the number of GPUs; over-provisioning processes wastes RAM (each loads the model).
6. **Scrape `/metrics` and alert on queue/duration.** Mosec exports per-stage batch size, queue time, and processing time — wire these into Prometheus/Grafana before you go live.

## Common Pitfalls <a id="pitfalls"></a>

1. **Forgetting `forward` gets a *list* when batched.** With `max_batch_size>1`, indexing `data["x"]` (treating it as one request) breaks. You must iterate/stack the list and return a list of equal length and order.
2. **Loading the model in `forward`.** Re-initializing weights every call destroys latency and may OOM. Always do it in `__init__`.
3. **Mismatched output length under batching.** The returned list must match the input batch length and order exactly, or responses get routed to the wrong clients.
4. **Setting `num` too high.** Each worker is a full process that loads its own copy of the model — large `num` on a big model exhausts host/GPU memory. Scale deliberately.
5. **Heavy work in `deserialize`/`serialize`.** These run in the worker too; expensive decoding here can bottleneck a stage. Keep them lean or push work into a dedicated stage.
6. **Expecting a non-Linux GPU experience.** Mosec's multiprocessing/shared-memory model targets Linux for production GPU serving; develop accordingly (containers help).

## Performance Optimization <a id="performance"></a>

### Configuration tuning

Key knobs, all set on `append_worker`:

- **`max_batch_size`** — upper bound on batch size. Raise until GPU utilization plateaus or latency budget is hit. The single most impactful throughput lever.
- **`max_wait_time`** (ms) — how long the batcher waits to fill a batch before dispatching a partial one. Lower = lower latency under light load; higher = fuller batches under bursty load.
- **`num`** — process replicas for a stage. Set GPU stages to your GPU count; set CPU stages near your core count. Each replica = one model copy in memory.
- **`timeout`** (`Server`/CLI `--timeout`) — per-request deadline in ms; raise it for slow/streaming models so valid requests aren't cut off.

### Throughput checklist

1. Move all non-model work (decode, tokenize, format) into separate CPU stages.
2. Enable batching on the GPU stage and grow `max_batch_size` while watching `nvidia-smi` utilization and p99 latency.
3. Switch tensor payloads to msgpack to cut (de)serialization time.
4. Confirm shared-memory IPC is doing the heavy lifting for large arrays (avoid re-encoding big tensors to JSON between stages).

In [ ]:
# Simple closed-loop throughput probe against a running Mosec server.
# Run this from a client machine/process; it reports requests/sec and mean latency.
import json
import time
import urllib.request


def benchmark(n: int = 500, url: str = "http://127.0.0.1:8000/inference",
              payload: dict | None = None) -> dict:
    payload = payload or {"x": 7}
    body = json.dumps(payload).encode()
    start = time.perf_counter()
    for _ in range(n):
        req = urllib.request.Request(
            url, data=body, headers={"Content-Type": "application/json"}, method="POST"
        )
        with urllib.request.urlopen(req) as resp:
            resp.read()
    elapsed = time.perf_counter() - start
    return {"requests": n, "seconds": round(elapsed, 3),
            "rps": round(n / elapsed, 1), "mean_ms": round(1000 * elapsed / n, 2)}

# For real batching numbers, fire concurrent requests (threads/asyncio) so the
# dynamic batcher actually has multiple in-flight requests to group.
# print(benchmark())   # run while a server is up


## Production Deployment <a id="deployment"></a>

### Docker deployment

Package the service script and its model. Mosec installs as a normal wheel, so the
image is just your Python deps plus `server.py`.

```dockerfile
FROM python:3.10-slim

# System deps your model needs (example: image libs). Keep this minimal.
RUN apt-get update && apt-get install -y --no-install-recommends libgomp1 \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt   # includes: mosec, torch, ...

COPY server.py .
COPY model/ ./model/

EXPOSE 8000
# Bind to all interfaces so the container is reachable.
CMD ["python", "server.py", "--address", "0.0.0.0", "--port", "8000"]
```

For GPU, base the image on an NVIDIA CUDA runtime image (e.g.
`nvidia/cuda:12.4.1-runtime-ubuntu22.04`), install a CUDA-enabled framework build, and
run the container with `--gpus all`.

### Kubernetes deployment

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mosec-embed
spec:
  replicas: 3
  selector:
    matchLabels: { app: mosec-embed }
  template:
    metadata:
      labels: { app: mosec-embed }
    spec:
      containers:
        - name: server
          image: registry.example.com/mosec-embed:1.0.0
          args: ["python", "server.py", "--address", "0.0.0.0", "--port", "8000"]
          ports:
            - containerPort: 8000
          resources:
            limits:
              nvidia.com/gpu: 1          # one GPU per pod (Inference stage num=1)
              memory: "8Gi"
            requests:
              cpu: "4"
              memory: "6Gi"
          readinessProbe:
            httpGet: { path: /, port: 8000 }
            initialDelaySeconds: 20      # allow model load in __init__
          livenessProbe:
            httpGet: { path: /, port: 8000 }
---
apiVersion: v1
kind: Service
metadata:
  name: mosec-embed
spec:
  selector: { app: mosec-embed }
  ports:
    - port: 80
      targetPort: 8000
```

Scale **horizontally with replicas/HPA** for more total capacity; tune `num` and
`max_batch_size` **inside each pod** for per-instance efficiency. Size the readiness
probe delay to cover model loading in `__init__`.

## Monitoring and Observability <a id="monitoring"></a>

### Key metrics to track

Mosec exposes Prometheus metrics at `GET /metrics`. The most actionable ones:

- **Batch size** — observed batch size per stage. If it sits near 1 under load, batching isn't helping (raise `max_wait_time`, check concurrency).
- **Queue / wait duration** — time requests spend waiting before a worker picks them up. Rising queue time = the stage is under-provisioned (raise `num` or batch size).
- **Processing duration** — time inside `forward` per stage. Pinpoints which stage is the bottleneck.
- **Throughput & error rate** — requests/sec and `4xx`/`5xx` counts (e.g. `ValidationError` → `422`, timeouts → `5xx`).

### Logging best practices

- Use Mosec's `get_logger()` so worker logs share the server's structured format and are attributable to the right stage/process.
- **Log what matters** — model version, batch sizes, and exceptions; avoid logging full payloads (PII, volume).
- **Use appropriate levels** — `INFO` for lifecycle (model loaded), `WARNING` for recoverable issues, `ERROR` with stack traces for failures.
- Ship logs and `/metrics` to your stack (Loki/ELK + Prometheus/Grafana) and **alert on queue time and error rate**, the two earliest signals of trouble.

## Troubleshooting <a id="troubleshooting"></a>

#### Issue 1: Throughput doesn't improve with batching

**Symptoms:** GPU under-utilized; observed batch size ≈ 1 in `/metrics`.

**Cause:** Requests arrive one at a time (no concurrency), or `max_wait_time` is too low to ever fill a batch.

**Solution:** Drive concurrent load from the client (threads/async); raise `max_wait_time` (e.g. 10–20ms) so the batcher can group requests; confirm `max_batch_size>1`.

#### Issue 2: Worker crashes / OOM on startup

**Symptoms:** Process dies during boot, or host/GPU memory exhausted as `num` rises.

**Cause:** Each of the `num` worker processes loads its **own** copy of the model. Large `num` × large model = OOM. Or weights are loaded in `forward` instead of `__init__`.

**Solution:** Lower `num`; size memory for `num × model_size`; load weights once in `__init__`. For GPU, ensure GPU stage `num` ≤ GPU count.

#### Issue 3: Requests time out under load

**Symptoms:** Clients get `5xx`/timeout errors when traffic spikes or for slow models.

**Cause:** Per-request `timeout` is shorter than the real processing+queue time, or a stage is under-provisioned and the queue backs up.

**Solution:** Raise `--timeout`; increase `num` on the bottleneck stage (identify it via per-stage processing duration in `/metrics`); add horizontal replicas.

#### Issue 4: Wrong responses returned to clients under batching

**Symptoms:** Clients occasionally receive another request's result.

**Cause:** `forward` returned a list whose length/order doesn't match the input batch.

**Solution:** Always return exactly one output per input, in the same order; never filter or reorder the batch inside `forward`.

## Comparison with Alternatives <a id="comparison"></a>

| Aspect | **Mosec** | Triton Inference Server | TorchServe | BentoML |
|--------|-----------|-------------------------|------------|---------|
| Core language | Rust runtime + Python workers | C++ | Java + Python | Python |
| Model format | Any Python code | Framework backends (TRT/ONNX/PyTorch/TF…) | TorchScript / eager `.mar` | Any Python code |
| Dynamic batching | ✅ built-in, per-stage | ✅ advanced | ✅ | ✅ |
| Multi-stage pipeline | ✅ first-class (`append_worker`) | ✅ (ensembles/BLS) | limited | ✅ (Runners/services) |
| LLM/SSE streaming | ✅ `send_stream_event` | partial | limited | ✅ |
| Ops surface | Minimal (a script) | Heavy, feature-rich | Medium | Medium, opinionated packaging |
| Best fit | Custom Python pipelines wanting batching + scaling, low overhead | Maximum-perf, multi-framework, GPU-dense fleets | PyTorch-centric shops | End-to-end "bento" packaging + deployment |

### When to choose Mosec

- You want **dynamic batching and multi-process scaling for arbitrary Python inference** without adopting a heavy server or converting model formats.
- You need **independently-scaled CPU and GPU stages** in one service.
- You value a **tiny, explicit API** and low runtime overhead over a large feature surface.

Prefer **Triton** when you need its breadth (multi-framework backends, model ensembles,
GPU-fleet features) and can absorb the complexity; prefer **vLLM/TGI** for
LLM-specialized serving with paged attention; prefer **BentoML** when you want
opinionated end-to-end packaging and deployment tooling.

## Resources <a id="resources"></a>

### Official documentation

- Documentation: https://mosecorg.github.io/mosec/
- GitHub repository: https://github.com/mosecorg/mosec
- PyPI: https://pypi.org/project/mosec/

### Tutorials and guides

- Getting started guide: https://mosecorg.github.io/mosec/
- Examples (PyTorch, embeddings, stable diffusion, LLM streaming): https://github.com/mosecorg/mosec/tree/main/examples
- API reference (Server / Worker / mixins): https://mosecorg.github.io/mosec/reference/

### Community resources

- Issues & discussions: https://github.com/mosecorg/mosec/issues
- Release notes / changelog: https://github.com/mosecorg/mosec/releases

### Related technologies

- NVIDIA Triton Inference Server — multi-framework, GPU-dense serving
- TorchServe — PyTorch-native serving
- BentoML — end-to-end model packaging and deployment
- vLLM / Text Generation Inference (TGI) — LLM-specialized serving with dynamic batching